#Assignment 2. ASR decoding

##Implementing required decoding methods:
+ Greedy decoding
+ Beam search decoding
+ Beam search with LM scores fusion
+ Beam search with a second pass LM rescoring

In [2]:
%%capture

!sudo apt-get update && apt-get upgrade && apt-get install cmake'
!pip install https://github.com/kpu/kenlm/archive/master.zip
!pip install levenshtein

In [3]:
from typing import List, Tuple

import kenlm
import heapq
import torch
import torchaudio
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC

import Levenshtein

In [30]:
import gzip
import shutil
import os

In [31]:
lm_gz_path = "models/3-gram.pruned.1e-7.arpa.gz"
lm_arpa_path = "models/3-gram.pruned.1e-7.arpa"

In [32]:
if not os.path.exists(lm_arpa_path):
    with gzip.open(lm_gz_path, 'rb') as f_in:
        with open(lm_arpa_path, 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)
    print("LM file was extracted.")
else:
    print("LM file already extracted.")

LM file was extracted.


In [ ]:
class Wav2Vec2Decoder:
    def __init__(
            self,
            model_name="facebook/wav2vec2-base-960h",
            lm_model_path="models/3-gram.pruned.1e-7.arpa",
            beam_width=3,
            alpha=1.0,
            beta=1.0
        ):
        """
        Initialization of Wav2Vec2Decoder class

        Args:
            model_name (str): Pretrained Wav2Vec2 model from transformers
            lm_model_path (str): Path to the KenLM n-gram model (for LM rescoring)
            beam_width (int): Number of hypotheses to keep in beam search
            alpha (float): LM weight for shallow fusion and rescoring
            beta (float): Word bonus for shallow fusion
        """
        # once logits are available, no other interactions with the model are allowed
        self.processor = Wav2Vec2Processor.from_pretrained(model_name)
        self.model = Wav2Vec2ForCTC.from_pretrained(model_name)

        # you can interact with these parameters
        self.vocab = {i: c for c, i in self.processor.tokenizer.get_vocab().items()}
        self.blank_token_id = self.processor.tokenizer.pad_token_id
        self.word_delimiter = self.processor.tokenizer.word_delimiter_token
        self.beam_width = beam_width
        self.alpha = alpha
        self.beta = beta
        self.lm_model = kenlm.Model(lm_model_path) if lm_model_path else None

    def _ctc_postprocess(self, token_ids: List[int]) -> str:
      """
      Remove repeats and blanks from token sequence (CTC decoding) and convert to text.

      Args:
        token_ids (List[int]): List of token IDs

      Returns:
        str: Decoded and cleaned transcript
      """
      filtered = [] #removing the doubled preds and getting rid of the blanks
      prev = None
      for i in token_ids:
          if i != prev and i != self.blank_token_id:
              filtered.append(i)
              prev = i
      tokens = [self.vocab.get(i, "") for i in filtered]
      return "".join(tokens).replace(self.word_delimiter, " ").strip() #to normal text

    def greedy_decode(self, logits: torch.Tensor) -> str:
        """
        Perform greedy decoding (find best CTC path)

        Args:
            logits (torch.Tensor): Logits from Wav2Vec2 model (T, V)

        Returns:
            str: Decoded transcript
        """

        pred_ids = torch.argmax(logits, dim=-1)
        return self._ctc_postprocess(pred_ids.tolist())

    def beam_search_decode(self, logits: torch.Tensor, return_beams: bool = False):
      """
        Perform beam search decoding (no LM)

        Args:
            logits (torch.Tensor): Logits from Wav2Vec2 model (T, V), where
                T - number of time steps and
                V - vocabulary size
            return_beams (bool): Return all beam hypotheses for second pass LM rescoring

        Returns:
            Union[str, List[Tuple[float, List[int]]]]:
                (str) - If return_beams is False, returns the best decoded transcript as a string.
                (List[Tuple[List[int], float]]) - If return_beams is True, returns a list of tuples
                    containing hypotheses and log probabilities.
        """
      log_probs = torch.log_softmax(logits, dim=-1) #remember to get log probabilities
      T, V = log_probs.shape
      beam_width = self.beam_width


      beams = [(0.0, [])]

      for t in range(T):
        new_beams = []
        for score, seq in beams:
            for c in range(V):
                new_seq = seq + [c]
                new_score = score + log_probs[t, c].item()
                new_beams.append((new_score, new_seq))

        # taking only the best ones in beam width area
        beams = heapq.nlargest(beam_width, new_beams, key=lambda x: x[0])

      if return_beams:
         return [(token_ids, score) for score, token_ids in beams]


      best_score, best_seq = beams[0]
      return self._ctc_postprocess(best_seq)

    def beam_search_with_lm(self, logits: torch.Tensor) -> str:
        """
        Perform beam search decoding with shallow LM fusion

        Args:
            logits (torch.Tensor): Logits from Wav2Vec2 model (T, V), where
                T - number of time steps and
                V - vocabulary size

        Returns:
            str: Decoded transcript
        """
        if not self.lm_model:
            raise ValueError("KenLM model required for LM shallow fusion")

        log_probs = torch.log_softmax(logits, dim=-1)
        T, V = log_probs.shape
        beam_width = self.beam_width

        alpha = self.alpha
        beta = self.beta


        beams = [(0.0, [])]

        for t in range(T):
            new_beams = []
            for score, seq in beams:
                for c in range(V):
                    new_seq = seq + [c]
                    ctc_score = score + log_probs[t, c].item()

                    decoded_text = "".join([self.vocab.get(i, "") for i in new_seq])
                    decoded_text = decoded_text.replace(self.word_delimiter, " ").strip()

                    lm_score = self.lm_model.score(decoded_text, bos=False, eos=False)
                    num_words = decoded_text.count(" ")
                    total_score = ctc_score + alpha * lm_score + beta * num_words

                    new_beams.append((total_score, new_seq))
            beams = heapq.nlargest(beam_width, new_beams, key=lambda x: x[0])

        _, best_seq = beams[0]
        return self._ctc_postprocess(best_seq)

    def lm_rescore(self, beams: List[Tuple[List[int], float]]) -> str:
        """
        Perform second-pass LM rescoring on beam search outputs

        Args:
            beams (list): List of tuples (hypothesis, log_prob)

        Returns:
            str: Best rescored transcript
        """
        if not self.lm_model:
            raise ValueError("KenLM model required for LM rescoring")

        best_score = float("-inf")
        best_text = ""

        for token_ids, ctc_score in beams:
            decoded_text = self._ctc_postprocess(token_ids)

            lm_score = self.lm_model.score(decoded_text, bos=False, eos=False)
            num_words = decoded_text.count(" ")
            total_score = ctc_score + self.alpha * lm_score + self.beta * num_words

            if total_score > best_score:
              best_score = total_score
              best_text = decoded_text


        return best_text

    def decode(self, audio_input: torch.Tensor, method: str = "greedy") -> str:
        """
        Decode input audio file using the specified method

        Args:
            audio_input (torch.Tensor): Audio tensor
            method (str): Decoding method ("greedy", "beam", "beam_lm", "beam_lm_rescore"),
                where "greedy" is a greedy decoding,
                      "beam" is beam search without LM,
                      "beam_lm" is beam search with LM shallow fusion, and
                      "beam_lm_rescore" is a beam search with second pass LM rescoring

        Returns:
            str: Decoded transcription
        """
        inputs = self.processor(audio_input, return_tensors="pt", sampling_rate=16000)
        with torch.no_grad():
            logits = self.model(inputs.input_values.squeeze(0)).logits[0]

        if method == "greedy":
            return self.greedy_decode(logits)
        elif method == "beam":
            return self.beam_search_decode(logits)
        elif method == "beam_lm":
            return self.beam_search_with_lm(logits)
        elif method == "beam_lm_rescore":
            beams = self.beam_search_decode(logits, return_beams=True)
            return self.lm_rescore(beams)
        else:
            raise ValueError("Invalid decoding method. Choose one of 'greedy', 'beam', 'beam_lm', 'beam_lm_rescore'.")


def test(decoder, audio_path, true_transcription):



    audio_input, sr = torchaudio.load(audio_path)
    assert sr == 16000, "Audio sample rate must be 16kHz"

    print("=" * 60)
    print("Target transcription")
    print(true_transcription)

    # Print all decoding methods results
    for d_strategy in ["greedy", "beam", "beam_lm", "beam_lm_rescore"]:
        print("-" * 60)
        print(f"{d_strategy} decoding")
        transcript = decoder.decode(audio_input, method=d_strategy)
        print(f"{transcript}")
        print(f"Character-level Levenshtein distance: {Levenshtein.distance(true_transcription, transcript.strip())}")


if __name__ == "__main__":

    test_samples = [
        ("examples/sample1.wav", "IF YOU ARE GENEROUS HERE IS A FITTING OPPORTUNITY FOR THE EXERCISE OF YOUR MAGNANIMITY IF YOU ARE PROUD HERE AM I YOUR RIVAL READY TO ACKNOWLEDGE MYSELF YOUR DEBTOR FOR AN ACT OF THE MOST NOBLE FORBEARANCE"),
        ("examples/sample2.wav", "AND IF ANY OF THE OTHER COPS HAD PRIVATE RACKETS OF THEIR OWN IZZY WAS UNDOUBTEDLY THE MAN TO FIND IT OUT AND USE THE INFORMATION WITH A BEAT SUCH AS THAT EVEN GOING HALVES AND WITH ALL THE GRAFT TO THE UPPER BRACKETS HE'D STILL BE ABLE TO MAKE HIS PILE IN A MATTER OF MONTHS"),
        ("examples/sample3.wav", "GUESS A MAN GETS USED TO ANYTHING HELL MAYBE I CAN HIRE SOME BUMS TO SIT AROUND AND WHOOP IT UP WHEN THE SHIPS COME IN AND BILL THIS AS A REAL OLD MARTIAN DEN OF SIN"),
        ("examples/sample4.wav", "IT WAS A TUNE THEY HAD ALL HEARD HUNDREDS OF TIMES SO THERE WAS NO DIFFICULTY IN TURNING OUT A PASSABLE IMITATION OF IT TO THE IMPROVISED STRAINS OF I DIDN'T WANT TO DO IT THE PRISONER STRODE FORTH TO FREEDOM"),
        ("examples/sample5.wav", "MARGUERITE TIRED OUT WITH THIS LONG CONFESSION THREW HERSELF BACK ON THE SOFA AND TO STIFLE A SLIGHT COUGH PUT UP HER HANDKERCHIEF TO HER LIPS AND FROM THAT TO HER EYES"),
        ("examples/sample6.wav", "AT THIS TIME ALL PARTICIPANTS ARE IN A LISTEN ONLY MODE"),
        ("examples/sample7.wav", "THE INCREASE WAS MAINLY ATTRIBUTABLE TO THE NET INCREASE IN THE AVERAGE SIZE OF OUR FLEETS"),
        ("examples/sample8.wav", "OPERATING SURPLUS IS A NON CAP FINANCIAL MEASURE WHICH IS DEFINED AS FULLY IN OUR PRESS RELEASE"),
    ]

    decoder = Wav2Vec2Decoder()

    _ = [test(decoder, audio_path, target) for audio_path, target in test_samples]

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Target transcription
IF YOU ARE GENEROUS HERE IS A FITTING OPPORTUNITY FOR THE EXERCISE OF YOUR MAGNANIMITY IF YOU ARE PROUD HERE AM I YOUR RIVAL READY TO ACKNOWLEDGE MYSELF YOUR DEBTOR FOR AN ACT OF THE MOST NOBLE FORBEARANCE
------------------------------------------------------------
greedy decoding
IF YOU ARE GENEROUS HERE IS A FITING OPORTUNITY FOR THE EXERCISE OF YOUR MAGNANIMITY IF YOU ARE PROUD HERE AM I YOUR RIVAL RETER TO ACKNOWLEDGE MYSELF YOUR DEPTOR FOR AN ACT OF MOST NOBLE FORBEARANCE
Character-level Levenshtein distance: 10
------------------------------------------------------------
beam decoding
IF YOU ARE GENEROUS HERE IS A FITING OPORTUNITY FOR THE EXERCISE OF YOUR MAGNANIMITY IF YOU ARE PROUD HERE AM I YOUR RIVAL RETER TO ACKNOWLEDGE MYSELF YOUR DEPTOR FOR AN ACT OF MOST NOBLE FORBEARANCE
Character-level Levenshtein distance: 10
------------------------------------------------------------
beam_lm decoding
IF YOU ARE GENEROUS HERE IS A FITING OPORTUNITY FOR THE EXERC

some quick observations:

+ The difference between greedy and beam strategies is minimal — in most cases, they produce identical Levenshtein distances. This may indicate that the beam width is too small (only 3) and doesn't provide a meaningful improvement.

+ In some examples, greedy performs as well as or even better than beam_lm, which is unusual and may result from over-weighting the LM score on short text fragments.

+ beam_lm sometimes performs worse than plain beam, suggesting suboptimal alpha and beta values, or low relevance of the LM to the generated hypotheses.

+ beam_lm_rescore usually returns to the quality of the plain beam. The second-pass LM rescoring stabilizes the result but does not improve it.

+ On short and simple samples, all decoding strategies behave similarly. On longer ones, differences are more noticeable, but the LM effect remains limited.

It would be reasonable to increase the beam width to 5–10 and rerun the experiments. This should yield more diverse hypotheses and strengthen LM influence. It's also worth tuning alpha and beta, as the current values (1.0) do not always balance CTC and LM contributions effectively.



##playing with params

In [5]:
import pandas as pd

In [13]:
class Wav2Vec2Decoder:
    def __init__(
            self,
            model_name="facebook/wav2vec2-base-960h",
            lm_model_path="models/3-gram.pruned.1e-7.arpa",
            beam_width=3,
            alpha=1.0,
            beta=1.0
        ):
        """
        Initialization of Wav2Vec2Decoder class

        Args:
            model_name (str): Pretrained Wav2Vec2 model from transformers
            lm_model_path (str): Path to the KenLM n-gram model (for LM rescoring)
            beam_width (int): Number of hypotheses to keep in beam search
            alpha (float): LM weight for shallow fusion and rescoring
            beta (float): Word bonus for shallow fusion
        """
        # once logits are available, no other interactions with the model are allowed
        self.processor = Wav2Vec2Processor.from_pretrained(model_name)
        self.model = Wav2Vec2ForCTC.from_pretrained(model_name)

        # you can interact with these parameters
        self.vocab = {i: c for c, i in self.processor.tokenizer.get_vocab().items()}
        self.blank_token_id = self.processor.tokenizer.pad_token_id
        self.word_delimiter = self.processor.tokenizer.word_delimiter_token
        self.beam_width = beam_width
        self.alpha = alpha
        self.beta = beta
        self.lm_model = kenlm.Model(lm_model_path) if lm_model_path else None

    def _ctc_postprocess(self, token_ids: List[int]) -> str:
      """
      Remove repeats and blanks from token sequence (CTC decoding) and convert to text.

      Args:
        token_ids (List[int]): List of token IDs

      Returns:
        str: Decoded and cleaned transcript
      """
      filtered = [] #removing the doubled preds and getting rid of the blanks
      prev = None
      for i in token_ids:
          if i != prev and i != self.blank_token_id:
              filtered.append(i)
              prev = i
      tokens = [self.vocab.get(i, "") for i in filtered]
      return "".join(tokens).replace(self.word_delimiter, " ").strip() #to normal text

    def greedy_decode(self, logits: torch.Tensor) -> str:
        """
        Perform greedy decoding (find best CTC path)

        Args:
            logits (torch.Tensor): Logits from Wav2Vec2 model (T, V)

        Returns:
            str: Decoded transcript
        """

        pred_ids = torch.argmax(logits, dim=-1)
        return self._ctc_postprocess(pred_ids.tolist())

    def beam_search_decode(self, logits: torch.Tensor, return_beams: bool = False):
      """
        Perform beam search decoding (no LM)

        Args:
            logits (torch.Tensor): Logits from Wav2Vec2 model (T, V), where
                T - number of time steps and
                V - vocabulary size
            return_beams (bool): Return all beam hypotheses for second pass LM rescoring

        Returns:
            Union[str, List[Tuple[float, List[int]]]]:
                (str) - If return_beams is False, returns the best decoded transcript as a string.
                (List[Tuple[List[int], float]]) - If return_beams is True, returns a list of tuples
                    containing hypotheses and log probabilities.
        """
      log_probs = torch.log_softmax(logits, dim=-1) #remember to get log probabilities
      T, V = log_probs.shape
      beam_width = self.beam_width


      beams = [(0.0, [])]

      for t in range(T):
        new_beams = []
        for score, seq in beams:
            for c in range(V):
                new_seq = seq + [c]
                new_score = score + log_probs[t, c].item()
                new_beams.append((new_score, new_seq))

        # taking only the best ones in beam width area
        beams = heapq.nlargest(beam_width, new_beams, key=lambda x: x[0])

      if return_beams:
         return [(token_ids, score) for score, token_ids in beams]


      best_score, best_seq = beams[0]
      return self._ctc_postprocess(best_seq)


    def beam_search_with_lm(self, logits: torch.Tensor) -> str:
        """
        Perform beam search decoding with shallow LM fusion

        Args:
            logits (torch.Tensor): Logits from Wav2Vec2 model (T, V), where
                T - number of time steps and
                V - vocabulary size

        Returns:
            str: Decoded transcript
        """
        if not self.lm_model:
            raise ValueError("KenLM model required for LM shallow fusion")

        log_probs = torch.log_softmax(logits, dim=-1)
        T, V = log_probs.shape
        beam_width = self.beam_width
        alpha = self.alpha
        beta = self.beta

        beams = [(0.0, [])]

        for t in range(T):
            new_beams = []
            for score, seq in beams:
                for c in range(V):
                    new_seq = seq + [c]
                    ctc_score = score + log_probs[t, c].item()

                    decoded_text = "".join([self.vocab.get(i, "") for i in new_seq])
                    decoded_text = decoded_text.replace(self.word_delimiter, " ").strip()

                    lm_score = self.lm_model.score(decoded_text, bos=False, eos=False)
                    num_words = decoded_text.count(" ")
                    total_score = ctc_score + alpha * lm_score + beta * num_words

                    new_beams.append((total_score, new_seq))
            beams = heapq.nlargest(beam_width, new_beams, key=lambda x: x[0])

        _, best_seq = beams[0]
        return self._ctc_postprocess(best_seq)


    def lm_rescore(self, beams: List[Tuple[List[int], float]]) -> str:
        """
        Perform second-pass LM rescoring on beam search outputs

        Args:
            beams (list): List of tuples (hypothesis, log_prob)

        Returns:
            str: Best rescored transcript
        """
        if not self.lm_model:
            raise ValueError("KenLM model required for LM rescoring")

        best_score = float("-inf")
        best_text = ""

        for token_ids, ctc_score in beams:
            decoded_text = self._ctc_postprocess(token_ids)

            lm_score = self.lm_model.score(decoded_text, bos=False, eos=False)
            num_words = decoded_text.count(" ")
            total_score = ctc_score + self.alpha * lm_score + self.beta * num_words

            if total_score > best_score:
              best_score = total_score
              best_text = decoded_text



        return best_text

    def decode(self, audio_input: torch.Tensor, method: str = "greedy") -> str:
        """
        Decode input audio file using the specified method

        Args:
            audio_input (torch.Tensor): Audio tensor
            method (str): Decoding method ("greedy", "beam", "beam_lm", "beam_lm_rescore"),
                where "greedy" is a greedy decoding,
                      "beam" is beam search without LM,
                      "beam_lm" is beam search with LM shallow fusion, and
                      "beam_lm_rescore" is a beam search with second pass LM rescoring

        Returns:
            str: Decoded transcription
        """
        inputs = self.processor(audio_input, return_tensors="pt", sampling_rate=16000)
        with torch.no_grad():
            logits = self.model(inputs.input_values.squeeze(0)).logits[0]

        if method == "greedy":
            return self.greedy_decode(logits)
        elif method == "beam":
            return self.beam_search_decode(logits)
        elif method == "beam_lm":
            return self.beam_search_with_lm(logits)
        elif method == "beam_lm_rescore":
            beams = self.beam_search_decode(logits, return_beams=True)
            return self.lm_rescore(beams)
        else:
            raise ValueError("Invalid decoding method. Choose one of 'greedy', 'beam', 'beam_lm', 'beam_lm_rescore'.")

    def decode_from_logits(self, logits, method="greedy"):
        if method == "greedy":
            return self.greedy_decode(logits)
        elif method == "beam":
            return self.beam_search_decode(logits)
        elif method == "beam_lm":
            return self.beam_search_with_lm(logits)
        elif method == "beam_lm_rescore":
            beams = self.beam_search_decode(logits, return_beams=True)
            return self.lm_rescore(beams)
        else:
            raise ValueError(f"Invalid method: {method}")


results = []

def test(decoder, audio_path, true_transcription):

    audio_input, sr = torchaudio.load(audio_path)
    assert sr == 16000, "Audio sample rate must be 16kHz"
    file_name = audio_path.split("/")[-1]


    for d_strategy in ["greedy", "beam", "beam_lm", "beam_lm_rescore"]:
        print("-" * 60)
        print(f"{d_strategy} decoding")
        transcript = decoder.decode(audio_input, method=d_strategy)
        print(f"{transcript}")
        print(f"Character-level Levenshtein distance: {Levenshtein.distance(true_transcription, transcript.strip())}")


if __name__ == "__main__":

    test_samples = [
        ("examples/sample1.wav", "IF YOU ARE GENEROUS HERE IS A FITTING OPPORTUNITY FOR THE EXERCISE OF YOUR MAGNANIMITY IF YOU ARE PROUD HERE AM I YOUR RIVAL READY TO ACKNOWLEDGE MYSELF YOUR DEBTOR FOR AN ACT OF THE MOST NOBLE FORBEARANCE"),
        ("examples/sample2.wav", "AND IF ANY OF THE OTHER COPS HAD PRIVATE RACKETS OF THEIR OWN IZZY WAS UNDOUBTEDLY THE MAN TO FIND IT OUT AND USE THE INFORMATION WITH A BEAT SUCH AS THAT EVEN GOING HALVES AND WITH ALL THE GRAFT TO THE UPPER BRACKETS HE'D STILL BE ABLE TO MAKE HIS PILE IN A MATTER OF MONTHS"),
        ("examples/sample3.wav", "GUESS A MAN GETS USED TO ANYTHING HELL MAYBE I CAN HIRE SOME BUMS TO SIT AROUND AND WHOOP IT UP WHEN THE SHIPS COME IN AND BILL THIS AS A REAL OLD MARTIAN DEN OF SIN"),
        ("examples/sample4.wav", "IT WAS A TUNE THEY HAD ALL HEARD HUNDREDS OF TIMES SO THERE WAS NO DIFFICULTY IN TURNING OUT A PASSABLE IMITATION OF IT TO THE IMPROVISED STRAINS OF I DIDN'T WANT TO DO IT THE PRISONER STRODE FORTH TO FREEDOM"),
        ("examples/sample5.wav", "MARGUERITE TIRED OUT WITH THIS LONG CONFESSION THREW HERSELF BACK ON THE SOFA AND TO STIFLE A SLIGHT COUGH PUT UP HER HANDKERCHIEF TO HER LIPS AND FROM THAT TO HER EYES"),
        ("examples/sample6.wav", "AT THIS TIME ALL PARTICIPANTS ARE IN A LISTEN ONLY MODE"),
        ("examples/sample7.wav", "THE INCREASE WAS MAINLY ATTRIBUTABLE TO THE NET INCREASE IN THE AVERAGE SIZE OF OUR FLEETS"),
        ("examples/sample8.wav", "OPERATING SURPLUS IS A NON CAP FINANCIAL MEASURE WHICH IS DEFINED AS FULLY IN OUR PRESS RELEASE"),
    ]



    beam_width_values = [3, 5, 10]
    alpha_values = [0.5, 1.0, 1.5]
    beta_values = [0.0, 0.5, 1.0]
    methods = ["greedy", "beam", "beam_lm", "beam_lm_rescore"]


    decoder = Wav2Vec2Decoder()

    logits_cache = {}
    for audio_path, target in test_samples:
        audio_input, sr = torchaudio.load(audio_path)
        inputs = decoder.processor(audio_input, return_tensors="pt", sampling_rate=16000)
        with torch.no_grad():
            logits = decoder.model(inputs.input_values.squeeze(0)).logits[0]
            file_name = audio_path.split("/")[-1]
            logits_cache[file_name] = {
                "logits": logits,
                "target": target
            }



    results = []

    for beam_width in beam_width_values:
        for alpha in alpha_values:
            for beta in beta_values:
                decoder.beam_width = beam_width
                decoder.alpha = alpha
                decoder.beta = beta


                for file_name, data in logits_cache.items():
                    logits = data["logits"]
                    target = data["target"]

                    for method in methods:

                        transcript = decoder.decode_from_logits(logits, method=method)
                        dist = Levenshtein.distance(target.strip(), transcript.strip())

                        results.append({
                            "file": file_name,
                            "method": method,
                            "beam_width": beam_width,
                            "alpha": alpha,
                            "beta": beta,
                            "levenshtein": dist,
                            "transcript": transcript.strip()
                        })



Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [15]:
df = pd.DataFrame(results)
pd.set_option('display.max_colwidth', None)
print(df)

            file           method  beam_width  alpha  beta  levenshtein  \
0    sample1.wav           greedy           3    0.5   0.0           10   
1    sample1.wav             beam           3    0.5   0.0           10   
2    sample1.wav          beam_lm           3    0.5   0.0           11   
3    sample1.wav  beam_lm_rescore           3    0.5   0.0           10   
4    sample2.wav           greedy           3    0.5   0.0            5   
..           ...              ...         ...    ...   ...          ...   
859  sample7.wav  beam_lm_rescore          10    1.5   1.0           19   
860  sample8.wav           greedy          10    1.5   1.0           15   
861  sample8.wav             beam          10    1.5   1.0           15   
862  sample8.wav          beam_lm          10    1.5   1.0           16   
863  sample8.wav  beam_lm_rescore          10    1.5   1.0           15   

                                                                                                   

In [18]:
grouped = df.groupby(["method", "beam_width", "alpha", "beta"])["levenshtein"].mean().reset_index()

pivot_avg = grouped.pivot_table(
    index=["beam_width", "alpha", "beta"],
    columns="method",
    values="levenshtein"
).reset_index()

pivot_avg


method,beam_width,alpha,beta,beam,beam_lm,beam_lm_rescore,greedy
0,3,0.5,0.0,10.0,10.500,10.000,10.0
1,3,0.5,0.5,10.0,10.000,10.000,10.0
2,3,0.5,1.0,10.0,9.750,10.000,10.0
3,3,1.0,0.0,10.0,11.000,10.000,10.0
4,3,1.0,0.5,10.0,10.875,10.000,10.0
5,3,1.0,1.0,10.0,10.625,10.000,10.0
6,3,1.5,0.0,10.0,12.625,10.000,10.0
7,3,1.5,0.5,10.0,12.125,10.000,10.0
8,3,1.5,1.0,10.0,11.875,10.000,10.0
9,5,0.5,0.0,10.0,10.500,10.000,10.0


+ The greedy method shows consistent results: Levenshtein distance is always 10.0, and it is not affected by any parameters.

+ The beam method (without LM) is also stable: the average distance is 10.0 for any value of beam_width.

+ The beam_lm_rescore method is nearly identical to beam: values range from 10.0 to 10.125, indicating minimal influence from LM rescoring.

+ The beam_lm method is sensitive to the alpha parameter.

+ With alpha = 0.5, results are comparable to other methods (around 10.0).

+ With alpha = 1.0 or higher, performance drops: average distance reaches up to 13.625.

+ Increasing beta slightly mitigates the degradation but has limited effect.

+ The beam_width parameter has no significant impact: beam hypotheses remain similar.

Conclusions:

+ beam_lm requires tuning of alpha and beta. The optimal range is alpha = 0.5, beta = 0.5–1.0.

+ beam_lm_rescore and beam produce stable results but no improvements.

+ LM starts to degrade performance when its weight (alpha) is too high.

+ To safely use LM, alpha should be limited to <= 0.5.

In [21]:
pivot_avg.to_csv("decoding_comparison.csv", index=False)


In [20]:
df.to_csv("first_results.csv", index=False)


 ## loading larger N-gram LM model

In [33]:
lm_gz_path = "models/4-gram.arpa.gz"
lm_arpa_path = "models/4-gram.arpa"


In [34]:
if not os.path.exists(lm_arpa_path):
    with gzip.open(lm_gz_path, 'rb') as f_in:
        with open(lm_arpa_path, 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)
    print("LM file was extracted.")
else:
    print("LM file already extracted.")


LM file was extracted.


In [23]:
class Wav2Vec2Decoder:
    def __init__(
            self,
            model_name="facebook/wav2vec2-base-960h",
            lm_model_path="models/4-gram.arpa",
            beam_width=3,
            alpha=1.0,
            beta=1.0
        ):
        """
        Initialization of Wav2Vec2Decoder class

        Args:
            model_name (str): Pretrained Wav2Vec2 model from transformers
            lm_model_path (str): Path to the KenLM n-gram model (for LM rescoring)
            beam_width (int): Number of hypotheses to keep in beam search
            alpha (float): LM weight for shallow fusion and rescoring
            beta (float): Word bonus for shallow fusion
        """
        # once logits are available, no other interactions with the model are allowed
        self.processor = Wav2Vec2Processor.from_pretrained(model_name)
        self.model = Wav2Vec2ForCTC.from_pretrained(model_name)

        # you can interact with these parameters
        self.vocab = {i: c for c, i in self.processor.tokenizer.get_vocab().items()}
        self.blank_token_id = self.processor.tokenizer.pad_token_id
        self.word_delimiter = self.processor.tokenizer.word_delimiter_token
        self.beam_width = beam_width
        self.alpha = alpha
        self.beta = beta
        self.lm_model = kenlm.Model(lm_model_path) if lm_model_path else None

    def _ctc_postprocess(self, token_ids: List[int]) -> str:
      """
      Remove repeats and blanks from token sequence (CTC decoding) and convert to text.

      Args:
        token_ids (List[int]): List of token IDs

      Returns:
        str: Decoded and cleaned transcript
      """
      filtered = [] #removing the doubled preds and getting rid of the blanks
      prev = None
      for i in token_ids:
          if i != prev and i != self.blank_token_id:
              filtered.append(i)
              prev = i
      tokens = [self.vocab.get(i, "") for i in filtered]
      return "".join(tokens).replace(self.word_delimiter, " ").strip() #to normal text

    def greedy_decode(self, logits: torch.Tensor) -> str:
        """
        Perform greedy decoding (find best CTC path)

        Args:
            logits (torch.Tensor): Logits from Wav2Vec2 model (T, V)

        Returns:
            str: Decoded transcript
        """

        pred_ids = torch.argmax(logits, dim=-1)
        return self._ctc_postprocess(pred_ids.tolist())

    def beam_search_decode(self, logits: torch.Tensor, return_beams: bool = False):
      """
        Perform beam search decoding (no LM)

        Args:
            logits (torch.Tensor): Logits from Wav2Vec2 model (T, V), where
                T - number of time steps and
                V - vocabulary size
            return_beams (bool): Return all beam hypotheses for second pass LM rescoring

        Returns:
            Union[str, List[Tuple[float, List[int]]]]:
                (str) - If return_beams is False, returns the best decoded transcript as a string.
                (List[Tuple[List[int], float]]) - If return_beams is True, returns a list of tuples
                    containing hypotheses and log probabilities.
        """
      log_probs = torch.log_softmax(logits, dim=-1) #remember to get log probabilities
      T, V = log_probs.shape
      beam_width = self.beam_width


      beams = [(0.0, [])]

      for t in range(T):
        new_beams = []
        for score, seq in beams:
            for c in range(V):
                new_seq = seq + [c]
                new_score = score + log_probs[t, c].item()
                new_beams.append((new_score, new_seq))

        # taking only the best ones in beam width area
        beams = heapq.nlargest(beam_width, new_beams, key=lambda x: x[0])

      if return_beams:
         return [(token_ids, score) for score, token_ids in beams]


      best_score, best_seq = beams[0]
      return self._ctc_postprocess(best_seq)


    def beam_search_with_lm(self, logits: torch.Tensor) -> str:
        """
        Perform beam search decoding with shallow LM fusion

        Args:
            logits (torch.Tensor): Logits from Wav2Vec2 model (T, V), where
                T - number of time steps and
                V - vocabulary size

        Returns:
            str: Decoded transcript
        """
        if not self.lm_model:
            raise ValueError("KenLM model required for LM shallow fusion")

        log_probs = torch.log_softmax(logits, dim=-1)
        T, V = log_probs.shape
        beam_width = self.beam_width
        # blank_id = self.blank_token_id
        alpha = self.alpha
        beta = self.beta
        # word_delim = self.word_delimiter

        beams = [(0.0, [])]

        for t in range(T):
            new_beams = []
            for score, seq in beams:
                for c in range(V):
                    new_seq = seq + [c]
                    ctc_score = score + log_probs[t, c].item()

                    decoded_text = "".join([self.vocab.get(i, "") for i in new_seq])
                    decoded_text = decoded_text.replace(self.word_delimiter, " ").strip()

                    lm_score = self.lm_model.score(decoded_text, bos=False, eos=False)
                    num_words = decoded_text.count(" ")
                    total_score = ctc_score + alpha * lm_score + beta * num_words

                    new_beams.append((total_score, new_seq))
            beams = heapq.nlargest(beam_width, new_beams, key=lambda x: x[0])

        _, best_seq = beams[0]
        return self._ctc_postprocess(best_seq)

    def lm_rescore(self, beams: List[Tuple[List[int], float]]) -> str:
        """
        Perform second-pass LM rescoring on beam search outputs

        Args:
            beams (list): List of tuples (hypothesis, log_prob)

        Returns:
            str: Best rescored transcript
        """
        if not self.lm_model:
            raise ValueError("KenLM model required for LM rescoring")

        best_score = float("-inf")
        best_text = ""

        for token_ids, ctc_score in beams:
            decoded_text = self._ctc_postprocess(token_ids)

            lm_score = self.lm_model.score(decoded_text, bos=False, eos=False)
            num_words = decoded_text.count(" ")
            total_score = ctc_score + self.alpha * lm_score + self.beta * num_words

            if total_score > best_score:
              best_score = total_score
              best_text = decoded_text


        return best_text

    def decode(self, audio_input: torch.Tensor, method: str = "greedy") -> str:
        """
        Decode input audio file using the specified method

        Args:
            audio_input (torch.Tensor): Audio tensor
            method (str): Decoding method ("greedy", "beam", "beam_lm", "beam_lm_rescore"),
                where "greedy" is a greedy decoding,
                      "beam" is beam search without LM,
                      "beam_lm" is beam search with LM shallow fusion, and
                      "beam_lm_rescore" is a beam search with second pass LM rescoring

        Returns:
            str: Decoded transcription
        """
        inputs = self.processor(audio_input, return_tensors="pt", sampling_rate=16000)
        with torch.no_grad():
            logits = self.model(inputs.input_values.squeeze(0)).logits[0]

        if method == "greedy":
            return self.greedy_decode(logits)
        elif method == "beam":
            return self.beam_search_decode(logits)
        elif method == "beam_lm":
            return self.beam_search_with_lm(logits)
        elif method == "beam_lm_rescore":
            beams = self.beam_search_decode(logits, return_beams=True)
            return self.lm_rescore(beams)
        else:
            raise ValueError("Invalid decoding method. Choose one of 'greedy', 'beam', 'beam_lm', 'beam_lm_rescore'.")


def test(decoder, audio_path, true_transcription):



    audio_input, sr = torchaudio.load(audio_path)
    assert sr == 16000, "Audio sample rate must be 16kHz"

    print("=" * 60)
    print("Target transcription")
    print(true_transcription)

    # Print all decoding methods results
    for d_strategy in ["greedy", "beam", "beam_lm", "beam_lm_rescore"]:
        print("-" * 60)
        print(f"{d_strategy} decoding")
        transcript = decoder.decode(audio_input, method=d_strategy)
        print(f"{transcript}")
        print(f"Character-level Levenshtein distance: {Levenshtein.distance(true_transcription, transcript.strip())}")


if __name__ == "__main__":

    test_samples = [
        ("examples/sample1.wav", "IF YOU ARE GENEROUS HERE IS A FITTING OPPORTUNITY FOR THE EXERCISE OF YOUR MAGNANIMITY IF YOU ARE PROUD HERE AM I YOUR RIVAL READY TO ACKNOWLEDGE MYSELF YOUR DEBTOR FOR AN ACT OF THE MOST NOBLE FORBEARANCE"),
        ("examples/sample2.wav", "AND IF ANY OF THE OTHER COPS HAD PRIVATE RACKETS OF THEIR OWN IZZY WAS UNDOUBTEDLY THE MAN TO FIND IT OUT AND USE THE INFORMATION WITH A BEAT SUCH AS THAT EVEN GOING HALVES AND WITH ALL THE GRAFT TO THE UPPER BRACKETS HE'D STILL BE ABLE TO MAKE HIS PILE IN A MATTER OF MONTHS"),
        ("examples/sample3.wav", "GUESS A MAN GETS USED TO ANYTHING HELL MAYBE I CAN HIRE SOME BUMS TO SIT AROUND AND WHOOP IT UP WHEN THE SHIPS COME IN AND BILL THIS AS A REAL OLD MARTIAN DEN OF SIN"),
        ("examples/sample4.wav", "IT WAS A TUNE THEY HAD ALL HEARD HUNDREDS OF TIMES SO THERE WAS NO DIFFICULTY IN TURNING OUT A PASSABLE IMITATION OF IT TO THE IMPROVISED STRAINS OF I DIDN'T WANT TO DO IT THE PRISONER STRODE FORTH TO FREEDOM"),
        ("examples/sample5.wav", "MARGUERITE TIRED OUT WITH THIS LONG CONFESSION THREW HERSELF BACK ON THE SOFA AND TO STIFLE A SLIGHT COUGH PUT UP HER HANDKERCHIEF TO HER LIPS AND FROM THAT TO HER EYES"),
        ("examples/sample6.wav", "AT THIS TIME ALL PARTICIPANTS ARE IN A LISTEN ONLY MODE"),
        ("examples/sample7.wav", "THE INCREASE WAS MAINLY ATTRIBUTABLE TO THE NET INCREASE IN THE AVERAGE SIZE OF OUR FLEETS"),
        ("examples/sample8.wav", "OPERATING SURPLUS IS A NON CAP FINANCIAL MEASURE WHICH IS DEFINED AS FULLY IN OUR PRESS RELEASE"),
    ]

    decoder = Wav2Vec2Decoder()

    _ = [test(decoder, audio_path, target) for audio_path, target in test_samples]

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Target transcription
IF YOU ARE GENEROUS HERE IS A FITTING OPPORTUNITY FOR THE EXERCISE OF YOUR MAGNANIMITY IF YOU ARE PROUD HERE AM I YOUR RIVAL READY TO ACKNOWLEDGE MYSELF YOUR DEBTOR FOR AN ACT OF THE MOST NOBLE FORBEARANCE
------------------------------------------------------------
greedy decoding
IF YOU ARE GENEROUS HERE IS A FITING OPORTUNITY FOR THE EXERCISE OF YOUR MAGNANIMITY IF YOU ARE PROUD HERE AM I YOUR RIVAL RETER TO ACKNOWLEDGE MYSELF YOUR DEPTOR FOR AN ACT OF MOST NOBLE FORBEARANCE
Character-level Levenshtein distance: 10
------------------------------------------------------------
beam decoding
IF YOU ARE GENEROUS HERE IS A FITING OPORTUNITY FOR THE EXERCISE OF YOUR MAGNANIMITY IF YOU ARE PROUD HERE AM I YOUR RIVAL RETER TO ACKNOWLEDGE MYSELF YOUR DEPTOR FOR AN ACT OF MOST NOBLE FORBEARANCE
Character-level Levenshtein distance: 10
------------------------------------------------------------
beam_lm decoding
IF YOU ARE GENEROUS HERE IS A FITING OPORTUNITY FOR THE EXERC

again, some observations(in comparison to the 3-gram model results) are here. this is also the place where I start thinking that something's wrong haha...

+ **greedy** and **beam search** show the results that are identical in both 3-gram and 4-gram cases. The LM does not affect them, as expected.

+ **beam_lm**: in the 4-gram setup, results do not improve and are sometimes worse than with the 3-gram model. On average, errors stay the same or increase.

+ **beam_lm_rescore**: in the 4-gram setup, results are identical to the 3-gram version in most cases.


Conclusions so far:

+ The 4-gram LM does not provide meaningful improvement over the 3-gram model.

+ In some cases, it degrades the performance of beam_lm, possibly due to overfitting or a mismatch between the LM and beam hypotheses.

+ beam_lm_rescore remains stable, but offers no consistent gain.

anyway, let's check some other params

In [24]:
class Wav2Vec2Decoder:
    def __init__(
            self,
            model_name="facebook/wav2vec2-base-960h",
            lm_model_path="models/4-gram.arpa",
            beam_width=3,
            alpha=1.0,
            beta=1.0
        ):
        """
        Initialization of Wav2Vec2Decoder class

        Args:
            model_name (str): Pretrained Wav2Vec2 model from transformers
            lm_model_path (str): Path to the KenLM n-gram model (for LM rescoring)
            beam_width (int): Number of hypotheses to keep in beam search
            alpha (float): LM weight for shallow fusion and rescoring
            beta (float): Word bonus for shallow fusion
        """
        # once logits are available, no other interactions with the model are allowed
        self.processor = Wav2Vec2Processor.from_pretrained(model_name)
        self.model = Wav2Vec2ForCTC.from_pretrained(model_name)

        # you can interact with these parameters
        self.vocab = {i: c for c, i in self.processor.tokenizer.get_vocab().items()}
        self.blank_token_id = self.processor.tokenizer.pad_token_id
        self.word_delimiter = self.processor.tokenizer.word_delimiter_token
        self.beam_width = beam_width
        self.alpha = alpha
        self.beta = beta
        self.lm_model = kenlm.Model(lm_model_path) if lm_model_path else None

    def _ctc_postprocess(self, token_ids: List[int]) -> str:
      """
      Remove repeats and blanks from token sequence (CTC decoding) and convert to text.

      Args:
        token_ids (List[int]): List of token IDs

      Returns:
        str: Decoded and cleaned transcript
      """
      filtered = [] #removing the doubled preds and getting rid of the blanks
      prev = None
      for i in token_ids:
          if i != prev and i != self.blank_token_id:
              filtered.append(i)
              prev = i
      tokens = [self.vocab.get(i, "") for i in filtered]
      return "".join(tokens).replace(self.word_delimiter, " ").strip() #to normal text

    def greedy_decode(self, logits: torch.Tensor) -> str:
        """
        Perform greedy decoding (find best CTC path)

        Args:
            logits (torch.Tensor): Logits from Wav2Vec2 model (T, V)

        Returns:
            str: Decoded transcript
        """

        pred_ids = torch.argmax(logits, dim=-1)
        return self._ctc_postprocess(pred_ids.tolist())

    def beam_search_decode(self, logits: torch.Tensor, return_beams: bool = False):
      """
        Perform beam search decoding (no LM)

        Args:
            logits (torch.Tensor): Logits from Wav2Vec2 model (T, V), where
                T - number of time steps and
                V - vocabulary size
            return_beams (bool): Return all beam hypotheses for second pass LM rescoring

        Returns:
            Union[str, List[Tuple[float, List[int]]]]:
                (str) - If return_beams is False, returns the best decoded transcript as a string.
                (List[Tuple[List[int], float]]) - If return_beams is True, returns a list of tuples
                    containing hypotheses and log probabilities.
        """
      log_probs = torch.log_softmax(logits, dim=-1) #remember to get log probabilities
      T, V = log_probs.shape
      beam_width = self.beam_width


      beams = [(0.0, [])]

      for t in range(T):
        new_beams = []
        for score, seq in beams:
            for c in range(V):
                new_seq = seq + [c]
                new_score = score + log_probs[t, c].item()
                new_beams.append((new_score, new_seq))

        # taking only the best ones in beam width area
        beams = heapq.nlargest(beam_width, new_beams, key=lambda x: x[0])

      if return_beams:
         return [(token_ids, score) for score, token_ids in beams]


      best_score, best_seq = beams[0]
      return self._ctc_postprocess(best_seq)


    def beam_search_with_lm(self, logits: torch.Tensor) -> str:
        """
        Perform beam search decoding with shallow LM fusion

        Args:
            logits (torch.Tensor): Logits from Wav2Vec2 model (T, V), where
                T - number of time steps and
                V - vocabulary size

        Returns:
            str: Decoded transcript
        """
        if not self.lm_model:
            raise ValueError("KenLM model required for LM shallow fusion")

        log_probs = torch.log_softmax(logits, dim=-1)
        T, V = log_probs.shape
        beam_width = self.beam_width
        alpha = self.alpha
        beta = self.beta

        beams = [(0.0, [])]

        for t in range(T):
            new_beams = []
            for score, seq in beams:
                for c in range(V):
                    new_seq = seq + [c]
                    ctc_score = score + log_probs[t, c].item()

                    decoded_text = "".join([self.vocab.get(i, "") for i in new_seq])
                    decoded_text = decoded_text.replace(self.word_delimiter, " ").strip()

                    lm_score = self.lm_model.score(decoded_text, bos=False, eos=False)
                    num_words = decoded_text.count(" ")
                    total_score = ctc_score + alpha * lm_score + beta * num_words

                    new_beams.append((total_score, new_seq))
            beams = heapq.nlargest(beam_width, new_beams, key=lambda x: x[0])

        _, best_seq = beams[0]
        return self._ctc_postprocess(best_seq)


    def lm_rescore(self, beams: List[Tuple[List[int], float]]) -> str:
        """
        Perform second-pass LM rescoring on beam search outputs

        Args:
            beams (list): List of tuples (hypothesis, log_prob)

        Returns:
            str: Best rescored transcript
        """
        if not self.lm_model:
            raise ValueError("KenLM model required for LM rescoring")

        best_score = float("-inf")
        best_text = ""

        for token_ids, ctc_score in beams:
            decoded_text = self._ctc_postprocess(token_ids)

            lm_score = self.lm_model.score(decoded_text, bos=False, eos=False)
            num_words = decoded_text.count(" ")
            total_score = ctc_score + self.alpha * lm_score + self.beta * num_words

            if total_score > best_score:
              best_score = total_score
              best_text = decoded_text



        return best_text

    def decode(self, audio_input: torch.Tensor, method: str = "greedy") -> str:
        """
        Decode input audio file using the specified method

        Args:
            audio_input (torch.Tensor): Audio tensor
            method (str): Decoding method ("greedy", "beam", "beam_lm", "beam_lm_rescore"),
                where "greedy" is a greedy decoding,
                      "beam" is beam search without LM,
                      "beam_lm" is beam search with LM shallow fusion, and
                      "beam_lm_rescore" is a beam search with second pass LM rescoring

        Returns:
            str: Decoded transcription
        """
        inputs = self.processor(audio_input, return_tensors="pt", sampling_rate=16000)
        with torch.no_grad():
            logits = self.model(inputs.input_values.squeeze(0)).logits[0]

        if method == "greedy":
            return self.greedy_decode(logits)
        elif method == "beam":
            return self.beam_search_decode(logits)
        elif method == "beam_lm":
            return self.beam_search_with_lm(logits)
        elif method == "beam_lm_rescore":
            beams = self.beam_search_decode(logits, return_beams=True)
            return self.lm_rescore(beams)
        else:
            raise ValueError("Invalid decoding method. Choose one of 'greedy', 'beam', 'beam_lm', 'beam_lm_rescore'.")

    def decode_from_logits(self, logits, method="greedy"):
        if method == "greedy":
            return self.greedy_decode(logits)
        elif method == "beam":
            return self.beam_search_decode(logits)
        elif method == "beam_lm":
            return self.beam_search_with_lm(logits)
        elif method == "beam_lm_rescore":
            beams = self.beam_search_decode(logits, return_beams=True)
            return self.lm_rescore(beams)
        else:
            raise ValueError(f"Invalid method: {method}")


results = []

def test(decoder, audio_path, true_transcription):

    audio_input, sr = torchaudio.load(audio_path)
    assert sr == 16000, "Audio sample rate must be 16kHz"
    file_name = audio_path.split("/")[-1]


    for d_strategy in ["greedy", "beam", "beam_lm", "beam_lm_rescore"]:
        print("-" * 60)
        print(f"{d_strategy} decoding")
        transcript = decoder.decode(audio_input, method=d_strategy)
        print(f"{transcript}")
        print(f"Character-level Levenshtein distance: {Levenshtein.distance(true_transcription, transcript.strip())}")


if __name__ == "__main__":

    test_samples = [
        ("examples/sample1.wav", "IF YOU ARE GENEROUS HERE IS A FITTING OPPORTUNITY FOR THE EXERCISE OF YOUR MAGNANIMITY IF YOU ARE PROUD HERE AM I YOUR RIVAL READY TO ACKNOWLEDGE MYSELF YOUR DEBTOR FOR AN ACT OF THE MOST NOBLE FORBEARANCE"),
        ("examples/sample2.wav", "AND IF ANY OF THE OTHER COPS HAD PRIVATE RACKETS OF THEIR OWN IZZY WAS UNDOUBTEDLY THE MAN TO FIND IT OUT AND USE THE INFORMATION WITH A BEAT SUCH AS THAT EVEN GOING HALVES AND WITH ALL THE GRAFT TO THE UPPER BRACKETS HE'D STILL BE ABLE TO MAKE HIS PILE IN A MATTER OF MONTHS"),
        ("examples/sample3.wav", "GUESS A MAN GETS USED TO ANYTHING HELL MAYBE I CAN HIRE SOME BUMS TO SIT AROUND AND WHOOP IT UP WHEN THE SHIPS COME IN AND BILL THIS AS A REAL OLD MARTIAN DEN OF SIN"),
        ("examples/sample4.wav", "IT WAS A TUNE THEY HAD ALL HEARD HUNDREDS OF TIMES SO THERE WAS NO DIFFICULTY IN TURNING OUT A PASSABLE IMITATION OF IT TO THE IMPROVISED STRAINS OF I DIDN'T WANT TO DO IT THE PRISONER STRODE FORTH TO FREEDOM"),
        ("examples/sample5.wav", "MARGUERITE TIRED OUT WITH THIS LONG CONFESSION THREW HERSELF BACK ON THE SOFA AND TO STIFLE A SLIGHT COUGH PUT UP HER HANDKERCHIEF TO HER LIPS AND FROM THAT TO HER EYES"),
        ("examples/sample6.wav", "AT THIS TIME ALL PARTICIPANTS ARE IN A LISTEN ONLY MODE"),
        ("examples/sample7.wav", "THE INCREASE WAS MAINLY ATTRIBUTABLE TO THE NET INCREASE IN THE AVERAGE SIZE OF OUR FLEETS"),
        ("examples/sample8.wav", "OPERATING SURPLUS IS A NON CAP FINANCIAL MEASURE WHICH IS DEFINED AS FULLY IN OUR PRESS RELEASE"),
    ]

    beam_width_values = [3, 5, 10]
    alpha_values = [0.5, 1.0, 1.5]
    beta_values = [0.0, 0.5, 1.0]
    methods = ["greedy", "beam", "beam_lm", "beam_lm_rescore"]


    decoder = Wav2Vec2Decoder()

    logits_cache = {}
    for audio_path, target in test_samples:
        audio_input, sr = torchaudio.load(audio_path)
        inputs = decoder.processor(audio_input, return_tensors="pt", sampling_rate=16000)
        with torch.no_grad():
            logits = decoder.model(inputs.input_values.squeeze(0)).logits[0]
            file_name = audio_path.split("/")[-1]
            logits_cache[file_name] = {
                "logits": logits,
                "target": target
            }



    results = []

    for beam_width in beam_width_values:
        for alpha in alpha_values:
            for beta in beta_values:
                decoder.beam_width = beam_width
                decoder.alpha = alpha
                decoder.beta = beta

                for file_name, data in logits_cache.items():
                    logits = data["logits"]
                    target = data["target"]

                    for method in methods:
                        transcript = decoder.decode_from_logits(logits, method=method)
                        dist = Levenshtein.distance(target.strip(), transcript.strip())

                        results.append({
                            "file": file_name,
                            "method": method,
                            "beam_width": beam_width,
                            "alpha": alpha,
                            "beta": beta,
                            "levenshtein": dist,
                            "transcript": transcript.strip()
                        })

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [25]:
df_4_gram = pd.DataFrame(results)
df_4_gram.to_csv("df_4_gram.csv", index=False)

In [27]:
df_4_gram = pd.DataFrame(results)
pd.set_option('display.max_colwidth', None)
print(df_4_gram)

            file           method  beam_width  alpha  beta  levenshtein  \
0    sample1.wav           greedy           3    0.5   0.0           10   
1    sample1.wav             beam           3    0.5   0.0           10   
2    sample1.wav          beam_lm           3    0.5   0.0           11   
3    sample1.wav  beam_lm_rescore           3    0.5   0.0           10   
4    sample2.wav           greedy           3    0.5   0.0            5   
..           ...              ...         ...    ...   ...          ...   
859  sample7.wav  beam_lm_rescore          10    1.5   1.0           19   
860  sample8.wav           greedy          10    1.5   1.0           15   
861  sample8.wav             beam          10    1.5   1.0           15   
862  sample8.wav          beam_lm          10    1.5   1.0           15   
863  sample8.wav  beam_lm_rescore          10    1.5   1.0           15   

                                                                                                   

In [28]:
grouped_df_4 = df_4_gram.groupby(["method", "beam_width", "alpha", "beta"])["levenshtein"].mean().reset_index()

pivot_avg_4 = grouped_df_4.pivot_table(
    index=["beam_width", "alpha", "beta"],
    columns="method",
    values="levenshtein"
).reset_index()

pivot_avg_4

method,beam_width,alpha,beta,beam,beam_lm,beam_lm_rescore,greedy
0,3,0.5,0.0,10.0,10.500,10.000,10.0
1,3,0.5,0.5,10.0,10.000,10.000,10.0
2,3,0.5,1.0,10.0,9.625,10.000,10.0
3,3,1.0,0.0,10.0,11.125,10.000,10.0
4,3,1.0,0.5,10.0,11.000,10.000,10.0
5,3,1.0,1.0,10.0,10.750,10.000,10.0
6,3,1.5,0.0,10.0,12.500,10.000,10.0
7,3,1.5,0.5,10.0,11.625,10.000,10.0
8,3,1.5,1.0,10.0,11.625,10.000,10.0
9,5,0.5,0.0,10.0,10.500,10.000,10.0


In [29]:
pivot_avg_4.to_csv("decoding_comparison_4_gram.csv", index=False)

what we can say about expiremnts on 3-gram vs. 4-gram models? :

+ The **greedy**, **beam**, and **beam_lm_rescore** methods behave identically across both models: the Levenshtein distance is consistently around 10.0 and does not depend on parameter values.

The differences appear only in the **beam_lm** method.

+ 4-gram model made some improvements, such as:
- With alpha = 0.5, beta = 1.0, the 4-gram model achieves a minimum distance of 9.625, compared to 9.75 for the 3-gram — a small but measurable improvement.

- With alpha = 1.0 and beta = 1.0, the 4-gram model also performs slightly better (10.75 vs 10.875 for 3-gram).

+ but there is also noticeable quality degradation:

- With alpha = 1.5, the quality of the 4-gram model also starts to decline, but:

The maximum distance is 12.875 (vs 13.625 for the 3-gram).

On average, the 4-gram shows less degradation at higher alpha values, especially when beam_width is large.

**Conclusion:**

+ the 4-gram model provides slightly better quality at low alpha values and slightly less degradation at high alpha values.

+ However, the improvements are moderate. Tuning of alpha and beta remains critical, especially when using the language model.

Switching from a 3-gram to a 4-gram model is justified if the LM is actively used (beam_lm), particularly in tasks where every character matters.